In [1]:
import pandas as pd

In [2]:
source_path = "../data/source/online_retail_II.xlsx"

sheet_data = pd.read_excel(
    source_path,
    sheet_name=None
)

In [3]:
# inspect the workbook

source_row_count = sum(len(df) for df in sheet_data.values())

print("Sheets:", list(sheet_data.keys()))
print("Source rows:", source_row_count)

Sheets: ['Year 2009-2010', 'Year 2010-2011']
Source rows: 1067371


In [4]:
sheet_data['Year 2009-2010'].head(1)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom


In [5]:
# Add source metadata

for sheet_name, df in sheet_data.items():
    df["source_sheet"] = sheet_name
    df["source_row_number"] = df.index + 2

    invoice_month = pd.to_datetime(
        df["InvoiceDate"],
        errors="coerce"
    )

    df["batch_id"] = (
        invoice_month.dt.strftime("%Y-%m")
        .fillna("unassigned")
    )

combined_df = pd.concat(
    sheet_data.values(),
    ignore_index=True
)

In [6]:
print("Combined rows:", len(combined_df))
print("Rows match source:", len(combined_df) == source_row_count)
print(
    "Unassigned rows:",
    (combined_df["batch_id"] == "unassigned").sum()
)

print(
    combined_df[
        ["source_sheet", "source_row_number", "InvoiceDate", "batch_id"]
    ].head()
)

Combined rows: 1067371
Rows match source: True
Unassigned rows: 0
     source_sheet  source_row_number         InvoiceDate batch_id
0  Year 2009-2010                  2 2009-12-01 07:45:00  2009-12
1  Year 2009-2010                  3 2009-12-01 07:45:00  2009-12
2  Year 2009-2010                  4 2009-12-01 07:45:00  2009-12
3  Year 2009-2010                  5 2009-12-01 07:45:00  2009-12
4  Year 2009-2010                  6 2009-12-01 07:45:00  2009-12


In [7]:
# Create monthly CSV batches

output_folder = "../data/monthly_batches"

for batch_id, batch_df in combined_df.groupby("batch_id"):
    output_path = (
        f"{output_folder}/online_retail_{batch_id}.csv"
    )

    batch_df.to_csv(
        output_path,
        index=False
    )

    print(batch_id, len(batch_df))

2009-12 45228
2010-01 31555
2010-02 29388
2010-03 41511
2010-04 34057
2010-05 35323
2010-06 39983
2010-07 33383
2010-08 33306
2010-09 42091
2010-10 59098
2010-11 78015
2010-12 65004
2011-01 35147
2011-02 27707
2011-03 36748
2011-04 29916
2011-05 37030
2011-06 36874
2011-07 39518
2011-08 35284
2011-09 50226
2011-10 60742
2011-11 84711
2011-12 25526


In [8]:
combined_df.groupby("batch_id").size().head()

batch_id
2009-12    45228
2010-01    31555
2010-02    29388
2010-03    41511
2010-04    34057
dtype: int64

In [9]:
# Complete validation

expected_counts = combined_df.groupby("batch_id").size()

actual_total_rows = 0
all_batches_valid = True

for batch_id, expected_rows in expected_counts.items():
    file_path = (
        f"{output_folder}/online_retail_{batch_id}.csv"
    )

    file_df = pd.read_csv(file_path)
    actual_rows = len(file_df)

    row_count_matches = actual_rows == expected_rows
    batch_id_matches = (
        file_df["batch_id"] == batch_id
    ).all()

    if not row_count_matches or not batch_id_matches:
        all_batches_valid = False
        print("Validation failed:", batch_id)

    actual_total_rows += actual_rows

print("Expected batch files:", len(expected_counts))
print("Validated batch files:", len(expected_counts))
print("Expected total rows:", source_row_count)
print("Actual total rows:", actual_total_rows)
print(
    "Total rows match:",
    actual_total_rows == source_row_count
)
print("All batch files valid:", all_batches_valid)

Expected batch files: 25
Validated batch files: 25
Expected total rows: 1067371
Actual total rows: 1067371
Total rows match: True
All batch files valid: True
